In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

In [2]:
# Function to extract Product Title
def get_title(soup):

    try:
        # Outer Tag Object
        title = soup.find("span", attrs={"id":'productTitle'})
        
        # Inner NavigatableString Object
        title_value = title.text

        # Title as a string value
        title_string = title_value.strip()

    except AttributeError:
        title_string = ""

    return title_string

# Function to extract Product Price
def get_price(soup):

    try:
        price = soup.find("span", attrs={'id':'priceblock_ourprice'}).string.strip()

    except AttributeError:

        try:
            # If there is some deal price
            price = soup.find("span", attrs={'id':'priceblock_dealprice'}).string.strip()

        except:
            price = ""

    return price

# Function to extract Product Rating
def get_rating(soup):

    try:
        rating = soup.find("i", attrs={'class':'a-icon a-icon-star a-star-4-5'}).string.strip()
    
    except AttributeError:
        try:
            rating = soup.find("span", attrs={'class':'a-icon-alt'}).string.strip()
        except:
            rating = ""	

    return rating

# Function to extract Number of User Reviews
def get_review_count(soup):
    try:
        review_count = soup.find("span", attrs={'id':'acrCustomerReviewText'}).string.strip()

    except AttributeError:
        review_count = ""	

    return review_count

# Function to extract Availability Status
def get_availability(soup):
    try:
        available = soup.find("div", attrs={'id':'availability'})
        available = available.find("span").string.strip()

    except AttributeError:
        available = "Not Available"	

    return available



In [4]:
if __name__ == '__main__':

    # add your user agent 
    HEADERS = ({'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36', 'Accept-Language': 'en-US, en;q=0.5'})

    # The webpage URL
    URL = "https://www.amazon.com/s?k=playstation+4&ref=nb_sb_noss_2"

    # HTTP Request
    webpage = requests.get(URL, headers=HEADERS)

    # Soup Object containing all data
    soup = BeautifulSoup(webpage.content, "html.parser")

    # Fetch links as List of Tag Objects
    links = soup.find_all("a", attrs={'class':'a-link-normal s-no-outline'})

    # Store the links
    links_list = []

    # Loop for extracting links from Tag Objects
    for link in links:
            links_list.append(link.get('href'))

    d = {"title":[], "price":[], "rating":[], "reviews":[],"availability":[]}
    
    # Loop for extracting product details from each link 
    for link in links_list:
        # Fix URL construction - check if link already starts with http
        if link.startswith('http'):
            product_url = link
        else:
            # Make sure link starts with / to avoid double slashes
            if not link.startswith('/'):
                link = '/' + link
            product_url = "https://www.amazon.com" + link
            
        new_webpage = requests.get(product_url, headers=HEADERS)

        new_soup = BeautifulSoup(new_webpage.content, "html.parser")

        # Function calls to display all necessary product information
        d['title'].append(get_title(new_soup))
        d['price'].append(get_price(new_soup))
        d['rating'].append(get_rating(new_soup))
        d['reviews'].append(get_review_count(new_soup))
        d['availability'].append(get_availability(new_soup))

    
    amazon_df = pd.DataFrame.from_dict(d)
    amazon_df['title'].replace('', np.nan, inplace=True)
    amazon_df = amazon_df.dropna(subset=['title'])
    amazon_df.to_csv("amazon_data.csv", header=True, index=False)

C:\Users\Prafulla\AppData\Local\Temp\ipykernel_20872\2621233789.py:51: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  amazon_df['title'].replace('', np.nan, inplace=True)


In [5]:
amazon_df

,title,price,rating,reviews,availability
0,PlayStation 4 Pro 1TB Console,,4.5 out of 5 stars,"(12,794)",Only 1 left in stock - order soon.
1,Sony PlayStation 4 Pro 1TB Console - Black (PS...,,4.5 out of 5 stars,"(4,158)",Not Available
2,500GB PlayStation 4 Console - Destiny The Take...,,4.6 out of 5 stars,(957),Not Available
3,PlayStation 4 Slim 1TB Limited Edition Console...,,4.5 out of 5 stars,(823),Only 2 left in stock - order soon.
4,Sony Computer Entertainment HW 1TB Gold Core -...,,4.7 out of 5 stars,(731),Only 1 left in stock - order soon.
5,Sony Computer Entertainment PS4 PRO Star Wars ...,,4.6 out of 5 stars,(693),Only 1 left in stock - order soon.
6,PlayStation 4 500GB Console - Uncharted 4 Limi...,,4.6 out of 5 stars,(722),Only 1 left in stock - order soon.
7,500GB PlayStation 4 Console - Limited Edition ...,,4.6 out of 5 stars,(608),Only 1 left in stock - order soon.
8,"AceGamer 2 Pack for PS4 Controller, Wireless C...",,4.4 out of 5 stars,"(6,740)",Not Available
9,"PS4 Controller Charger Dock Station, OIVO 1.8H...",,4.6 out of 5 stars,"(43,862)",Not Available
